In [1]:
pip install azure-ai-projects azure-core azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 3.9 MB/s eta 0:00:00


In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.31.0
    Uninstalling openai-2.31.0:
      Successfully uninstalled openai-2.31.0


In [6]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("New_Open_AI")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://bolob435.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [9]:
# 1. Define the input data
ai_research_data = """
Title: AI-Based Healthcare Diagnosis System

Abstract:
This research paper explores the use of Artificial Intelligence in early disease detection.
Machine learning models are trained on patient datasets to predict diseases such as diabetes and cancer.

Introduction:
Artificial Intelligence is transforming healthcare by enabling faster and more accurate diagnosis.
This study focuses on predictive modeling techniques.

Methodology:
We used supervised learning algorithms including Random Forest and Neural Networks.
The dataset was preprocessed and split into training and testing sets.

Results:
The model achieved an accuracy of 92% on the test dataset.
Neural Networks performed better compared to traditional models.

Conclusion:
AI can significantly improve early diagnosis and reduce human error in healthcare systems.
"""

# 2. Extract structured research data
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """Extract the following details from the research paper and return ONLY valid JSON:

                        - Title
                        - Abstract (short summary)
                        - Methodology
                        - Key Results
                        - Conclusion
                        """
                    },
                    {
                        "type": "input_text",
                        "text": f"RESEARCH PAPER:\n{ai_research_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED RESEARCH DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED RESEARCH DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Title": "AI-Based Healthcare Diagnosis System",\n  "Abstract": "This research paper explores the use of Artificial Intelligence in early disease detection. Machine learning models are trained on patient datasets to predict diseases such as diabetes and cancer.",\n  "Methodology": "We used supervised learning algorithms including Random Forest and Neural Networks. The dataset was preprocessed and split into training and testing sets.",\n  "Key Results": "The model achieved an accuracy of 92% on the test dataset. Neural Networks performed better compared to traditional models.",\n  "Conclusion": "AI can significantly improve early diagnosis and reduce human error in healthcare systems."\n}', type='output_text', logprobs=[])]


In [10]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Title": "AI-Based Healthcare Diagnosis System",
  "Abstract": "This research paper explores the use of Artificial Intelligence in early disease detection. Machine learning models are trained on patient datasets to predict diseases such as diabetes and cancer.",
  "Methodology": "We used supervised learning algorithms including Random Forest and Neural Networks. The dataset was preprocessed and split into training and testing sets.",
  "Key Results": "The model achieved an accuracy of 92% on the test dataset. Neural Networks performed better compared to traditional models.",
  "Conclusion": "AI can significantly improve early diagnosis and reduce human error in healthcare systems."
}


In [11]:
from azure.storage.blob import BlobServiceClient

# ✅ Correct connection string
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=mrinalstorage;AccountKey=PHtgo0Sd+nYgLm4WXvHBp+rv7laJJsvg6Y3EXfO4ZEYuNDlffd2RdauObtclMrVVbxHpF6o8QmKN+ASt0qj8PA==;EndpointSuffix=core.windows.net"

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "researchop"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [12]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [13]:
blob_client = blob_service_client.get_blob_client(
    container="researchop",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F76D97117E9"',
 'last_modified': datetime.datetime(2026, 4, 21, 7, 23, 13, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'C)b\xdd\xf7w\x10udnG\xda8\xfb\x99\xec'),
 'client_request_id': 'f52db2b6-3d52-11f1-bdf7-0242ac1c000c',
 'request_id': '458cbf28-701e-00b3-0a5f-d12bca000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 7, 23, 12, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [14]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260421_072305.txt
